In [14]:
import numpy as np
import pandas as pd
from classy import Class
from getdist import loadMCSamples
from tqdm import tqdm

In [ ]:
data_file = "/home/panda/class_monte/cobaya/quintom-project/sources/data/sn_data/PantheonPlus/Pantheon+SH0ES.dat"
cov_file = "/home/panda/class_monte/cobaya/quintom-project/sources/data/sn_data/PantheonPlus/Pantheon+SH0ES_STAT+SYS.cov"

sn_data = pd.read_csv(data_file, delim_whitespace=True)

/tmp/ipykernel_2566456/3912088260.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  sn_data = pd.read_csv(data_file, delim_whitespace=True)


0       1
1       1
2       1
3       1
4       0
       ..
1696    0
1697    0
1698    0
1699    0
1700    0
Name: IS_CALIBRATOR, Length: 1701, dtype: int64

In [ ]:
# create a boolean mask to drop SH0ES calibrators (since we didn't use them)
mask = sn_data['IS_CALIBRATOR'] == 0
sn_data = sn_data[mask]

# Apply the mask to get the correct redshift and magnitude vectors
z_obs = sn_data.loc[mask, 'zHD'].values
m_obs = sn_data.loc[mask, 'm_b_corr'].values

In [12]:
with open(cov_file, 'r') as f:
    dim = int(f.readline().strip())
cov_matrix = np.loadtxt(cov_file, skiprows=1).reshape((dim, dim))
cov_matrix = cov_matrix[mask][:, mask]
inv_cov_matrix = np.linalg.inv(cov_matrix)
A = np.sum(inv_cov_matrix)
sigma_Mb = 1/np.sqrt(A)

In [23]:
chain_file = '/home/panda/class_monte/cobaya/quintom-project/chains/quintom/DESI+CMB+SNIa/quintom_DESI+CMB+SNIa'
samples = loadMCSamples(chain_file, settings={'ignore_rows': 0.3})

p = samples.getParams()
total_samples = len(samples.samples)
Mb_samples = np.zeros(total_samples)

In [30]:
cosmo = Class()

for i in tqdm(range(total_samples)):
    params = {
        'H0': p.H0[i],
        'omega_b': p.omega_b[i],
        'omega_cdm': p.omega_cdm[i],
    }
    try: 
        cosmo.set(params)
        cosmo.compute()
        lum_dist = np.array([cosmo.luminosity_distance(z) for z in z_obs])
        mu_th = 5 * np.log10(lum_dist) + 25

        delta = m_obs - mu_th
        B = np.sum(inv_cov_matrix @ delta)
        mean_Mb = B / A
        Mb_samples[i] = np.random.normal(loc=mean_Mb, scale=sigma_Mb)
    except Exception as e:
        print(f"Error at sample {i}: {e}")
        Mb_samples[i] = np.nan
    finally:
        cosmo.struct_cleanup()
        cosmo.empty()

100%|██████████| 35677/35677 [55:39<00:00, 10.68it/s]  


In [31]:
valid_indices = ~np.isnan(Mb_samples)
if not np.all(valid_indices):
    samples.filter(valid_indices)
    Mb_samples = Mb_samples[valid_indices]

samples.addDerived(Mb_samples, name='Mb', label='M_b')
out_root = '/home/panda/class_monte/cobaya/quintom-project/chains/quintom_DESI+CMB+SNIa_with_Mb'
samples.saveAsText(out_root)